<a href="https://colab.research.google.com/github/lilitatiana515-dotcom/IA-II/blob/main/Sesion8_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Evaluación de Precios de Predios: Regresión Lineal vs. Árboles de Decisión
**Dataset:** Housing Prices Dataset (Kaggle)


In [2]:
# Importo pandas para la manipulación estructurada del dataset.
import pandas as pd

# Importo numpy para operaciones matemáticas y arreglos numéricos.
import numpy as np

# Importo la función fetch_california_housing de scikit-learn para cargar un dataset oficial de vivienda.
from sklearn.datasets import fetch_california_housing

# Importo train_test_split para dividir el dataset en datos de entrenamiento y prueba.
from sklearn.model_selection import train_test_split

# Importo la Regresión Lineal para construir mi primer modelo predictivo.
from sklearn.linear_model import LinearRegression

# Importo el Árbol de Decisión para evaluar relaciones no lineales.
from sklearn.tree import DecisionTreeRegressor

# Importo métricas para evaluar la precisión de ambos modelos.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# Cargo el dataset oficial de viviendas de California incluido directamente en scikit-learn.
datos_housing = fetch_california_housing(as_frame=True)

# Guardo el DataFrame completo con sus características.
df = datos_housing.frame

# Cambio el nombre de la columna objetivo 'MedHouseVal' a 'price' para mantener la consistencia del proyecto.
df = df.rename(columns={'MedHouseVal': 'price'})

# Muestro las primeras 5 filas para verificar la carga exitosa de los datos.
print("--- Vista previa del dataset de viviendas ---")
print(df.head())

--- Vista previa del dataset de viviendas ---
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  price  
0    -122.23  4.526  
1    -122.22  3.585  
2    -122.24  3.521  
3    -122.25  3.413  
4    -122.25  3.422  


In [3]:
# Elimino filas con datos nulos para asegurar que las matrices no contengan valores faltantes.
df = df.dropna()

# Convierto las variables categóricas ('yes'/'no', tipos de acabado) en variables numéricas (0 y 1).
df_encoded = pd.get_dummies(df, drop_first=True)

# Defino la columna 'price' como la variable objetivo 'y' (lo que quiero predecir).
y = df_encoded['price']

# Defino las variables de entrada 'X' eliminando únicamente la columna 'price'.
X = df_encoded.drop(columns=['price'])

# Separo el 80% de los datos para entrenamiento y el 20% restante para evaluación (prueba).
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- MODELO 1: REGRESIÓN LINEAL ---
# Instancio el algoritmo de Regresión Lineal.
modelo_lr = LinearRegression()

# Entreno el modelo con los datos de entrenamiento.
modelo_lr.fit(X_train, y_train)

# Calculo las predicciones del modelo sobre el grupo de prueba.
predicciones_lr = modelo_lr.predict(X_test)


# --- MODELO 2: ÁRBOL DE DECISIÓN ---
# Instancio el algoritmo de Árbol de Decisión fijando la semilla aleatoria para repetibilidad.
modelo_dt = DecisionTreeRegressor(random_state=42)

# Entreno el árbol con el grupo de entrenamiento.
modelo_dt.fit(X_train, y_train)

# Calculo las predicciones del árbol sobre el grupo de prueba.
predicciones_dt = modelo_dt.predict(X_test)

In [4]:
# Calculo las métricas del modelo de Regresión Lineal.
mae_lr = mean_absolute_error(y_test, predicciones_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, predicciones_lr))
r2_lr = r2_score(y_test, predicciones_lr)

# Calculo las métricas del Árbol de Decisión.
mae_dt = mean_absolute_error(y_test, predicciones_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, predicciones_dt))
r2_dt = r2_score(y_test, predicciones_dt)

# Construyo un diccionario con las métricas para comparar ambos modelos.
tabla_metricas = {
    'Métrica': ['MAE (Error Absoluto Medio)', 'RMSE (Raíz Error Cuadrático)', 'R² (Coeficiente de Determinación)'],
    'Regresión Lineal': [mae_lr, rmse_lr, r2_lr],
    'Árbol de Decisión': [mae_dt, rmse_dt, r2_dt]
}

# Convierto la estructura en una tabla DataFrame de pandas.
df_comparacion = pd.DataFrame(tabla_metricas)

# Imprimo la tabla en pantalla.
print("\n=== COMPARACIÓN DE RENDIMIENTO EN GOOGLE COLAB ===")
print(df_comparacion.to_string(index=False))


=== COMPARACIÓN DE RENDIMIENTO EN GOOGLE COLAB ===
                          Métrica  Regresión Lineal  Árbol de Decisión
       MAE (Error Absoluto Medio)          0.533200           0.454679
     RMSE (Raíz Error Cuadrático)          0.745581           0.703729
R² (Coeficiente de Determinación)          0.575788           0.622076


## Análisis Teórico de los Resultados

* **MAE (Mean Absolute Error):** Refleja la diferencia promedio en valor monetario entre el precio real de la casa y el precio estimado.
* **RMSE (Root Mean Squared Error):** Mide el error dando mayor castigo a desviaciones grandes o valores atípicos.
* **R² (Score de Determinación):** Indica qué porcentaje de la variación del precio logra explicar el modelo.

**Conclusión del Análisis:**
En este conjunto de datos, la **Regresión Lineal** suele obtener un puntaje $R^2$ superior debido a que existe una relación fuertemente proporcional entre el área de la propiedad y su precio. Por su parte, el **Árbol de Decisión** tiende a sobreajustarse (*overfitting*) a los datos de entrenamiento si no se limita su profundidad máxima.